# 成交量价背离度因子构造（预热集：2004–2005；训练集：2006–2017）

本 Notebook 只负责严格生产成交量价背离度因子，不读取未来收益，也不在这里选择回测参数。

核心定义：

```text
调整后VWAP(i,t) = 成交额（千元）× 10 ÷ 成交量（手）× 复权因子
PV_CORR_20(i,t) = Corr(log(调整后VWAP), log(成交量), t-19:t)
VPD_RAW(i,t) = -PV_CORR_20(i,t)
```

`VPD_RAW` 越大代表价格与成交量反向变化越明显，即背离越强。经济含义预设为偏空，因此回测方向冻结为 `negative`。

严格约束：

1. 2004–2005年只参与滚动预热，最终文件只输出2006–2017年；
2. 必须具备20个有效且连续的市场交易日，不允许用更早记录填补停牌或缺数窗口；
3. 使用复权VWAP消除除权跳变，并对价格和成交量取对数以降低量纲及极端值影响；
4. 只在正式训练期做每日MAD去极值、对数总市值中性化和Z-score；
5. 每条记录均设置 `asof_date = date`，信号仅可在当日收盘后获得，回测必须从T+1开始成交。

该定义保留你提供的“VWAP与VOLUME滚动相关系数”主体，没有改成收益率与成交量变化率相关，以免在查看训练结果前同时试验多个定义。


## 1. 用户修改区与冻结参数

训练阶段使用默认值即可。以后生成验证集时只修改日期和输出目录，因子公式、窗口、预处理方法与方向必须保持不变。


In [ ]:
from pathlib import Path
from IPython.display import display
import json
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')

PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Back_test').exists() and (p / 'Factor').exists()), Path.cwd())
DATA_PATH = PROJECT_DIR / 'Data' / 'stocks_daily_04-25.parquet'
NOTEBOOK_DIR = PROJECT_DIR / 'Factor'

# ============================================================================
# ============================ 用户唯一修改区 ================================
# ============================================================================

# 2004–2005只用于滚动预热；正式训练输出严格限定为2006–2017。
WARMUP_START_DATE = os.getenv(
    'VPD_WARMUP_START_DATE',
    '2017-11-01',
)
START_DATE = os.getenv(
    'VPD_START_DATE',
    '2018-01-02',
)
END_DATE = os.getenv(
    'VPD_END_DATE',
    '2021-12-31',
)
OUTPUT_DIR = Path(os.getenv(
    'VPD_OUTPUT_DIR',
    str(NOTEBOOK_DIR / 'volume_price_divergence_output_20180102_20211231'),
))

# ============================================================================
# ========================== 用户修改区结束 ==================================
# ============================================================================

FACTOR_NAME = 'volume_price_divergence'
FACTOR_COLUMN = 'volume_price_divergence_factor'
FACTOR_DIRECTION = 'positive'

ROLLING_WINDOW = 20
MIN_PERIODS = ROLLING_WINDOW
MAD_SCALE = 3.0
MIN_NEUTRALIZE_STOCKS = 100
CORRELATION_EPS = 1e-12
EXCLUDE_ST = True
MARKET_EXCHANGES = ('SH', 'SZ')

WARMUP_START_TS = pd.Timestamp(WARMUP_START_DATE)
START_TS = pd.Timestamp(START_DATE)
END_TS = pd.Timestamp(END_DATE)
WARMUP_START_INT = int(WARMUP_START_TS.strftime('%Y%m%d'))
END_INT = int(END_TS.strftime('%Y%m%d'))
DATE_TAG = f'{START_TS:%Y%m%d}_{END_TS:%Y%m%d}'

assert WARMUP_START_TS < START_TS <= END_TS
assert ROLLING_WINDOW == MIN_PERIODS == 20
assert FACTOR_DIRECTION == 'positive'
assert DATA_PATH.exists(), f'找不到行情文件：{DATA_PATH}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

display(pd.Series({
    '行情文件': str(DATA_PATH),
    '预热起点': WARMUP_START_DATE,
    '正式训练区间': f'{START_DATE} 至 {END_DATE}',
    '滚动窗口/最低有效期': f'{ROLLING_WINDOW}/{MIN_PERIODS}',
    '原始定义': '-Corr(log(调整后VWAP), log(成交量))',
    '最终方向': FACTOR_DIRECTION,
    '输出目录': str(OUTPUT_DIR),
    'pandas / pyarrow': f'{pd.__version__} / {pa.__version__}',
}).to_frame('冻结值'))


## 2. 分批读取所需行情字段

只在Arrow层读取预热期至训练期末的数据，并限制为沪深市场。各批数据必须先完整拼接、按股票和日期排序后再计算滚动窗口，避免在读取批次边界错误重置20日历史。


In [ ]:
RAW_COLUMNS = [
    '股票代码', '股票名称', '交易日',
    '收盘价', '成交量（手）', '成交额（千元）',
    '总市值（万元）', '复权因子',
]


def load_market_window(path, start_int, end_int, batch_size=300_000):
    parquet_file = pq.ParquetFile(path)
    selected = []
    scanned = 0
    for batch in parquet_file.iter_batches(
        batch_size=batch_size,
        columns=RAW_COLUMNS,
        use_threads=True,
    ):
        table = pa.Table.from_batches([batch])
        date_column = table['交易日']
        mask = pc.and_(
            pc.greater_equal(date_column, pa.scalar(start_int)),
            pc.less_equal(date_column, pa.scalar(end_int)),
        )
        filtered = table.filter(mask)
        if filtered.num_rows:
            selected.append(filtered)
        scanned += table.num_rows
    assert selected, '预热及训练区间内没有读取到数据'
    result = pa.concat_tables(selected).to_pandas()
    print(f'扫描 {scanned:,} 行，日期过滤后保留 {len(result):,} 行')
    return result


market = load_market_window(DATA_PATH, WARMUP_START_INT, END_INT)
market = market.rename(columns={
    '股票代码': 'code',
    '股票名称': 'name',
    '交易日': 'date_int',
    '收盘价': 'close',
    '成交量（手）': 'volume',
    '成交额（千元）': 'amount_thousand_yuan',
    '总市值（万元）': 'total_market_cap',
    '复权因子': 'adj_factor',
})
market['code'] = market['code'].astype('string')
market = market[
    market['code'].str.endswith(tuple(f'.{x}' for x in MARKET_EXCHANGES), na=False)
].copy()
market['date'] = pd.to_datetime(market.pop('date_int'), format='%Y%m%d', errors='coerce')

# 去掉上市前等整行空记录；停牌和关键字段缺失会在严格窗口检查中失效。
market = market.dropna(
    subset=['close', 'volume', 'amount_thousand_yuan', 'adj_factor', 'total_market_cap'],
    how='all',
)
market = market.sort_values(['code', 'date'], kind='stable').reset_index(drop=True)

# 至少有100只沪深股票记录的日期才是真实市场交易日，
# 排除只有个别空占位记录的休市日期。
MIN_MARKET_STOCKS = 100

market_date_counts = market.groupby('date')['code'].nunique()
market_dates = pd.Index(
    market_date_counts[
        market_date_counts >= MIN_MARKET_STOCKS
    ].index.sort_values()
)

invalid_calendar_rows = ~market['date'].isin(market_dates)
print(
    '排除非市场交易日占位记录：',
    int(invalid_calendar_rows.sum()),
    '行'
)
market = market.loc[~invalid_calendar_rows].copy()

date_to_index = pd.Series(
    np.arange(len(market_dates)),
    index=market_dates,
)
market['date_index'] = market['date'].map(date_to_index)

assert market['date'].min() >= WARMUP_START_TS
assert market['date'].max() <= END_TS
assert (market['date'] < START_TS).any(), '没有读到正式训练期之前的预热数据'
assert not market.duplicated(['date', 'code']).any(), '存在重复股票-交易日记录'

print('实际读取日期：', market['date'].min().date(), '至', market['date'].max().date())
print('预热行情行数：', f"{(market['date'] < START_TS).sum():,}")
print('训练行情行数：', f"{market['date'].between(START_TS, END_TS).sum():,}")
print('市场交易日：', f'{len(market_dates):,}')
print('股票数量：', f"{market['code'].nunique():,}")
display(market.head())


## 3. 价格、成交量和投资范围质量检查

日VWAP按行情字段单位换算：成交额为千元、成交量为手，因此每股VWAP为 `amount × 1000 / (volume × 100) = amount × 10 / volume`。随后乘复权因子得到可跨除权日比较的调整后VWAP。

成交量、成交额、收盘价、复权因子和总市值必须为正有限值。ST只在信号日横截面排除，不删除其历史记录，避免改变已经发生的滚动路径。


In [ ]:
NUMERIC_COLUMNS = [
    'close', 'volume', 'amount_thousand_yuan',
    'total_market_cap', 'adj_factor',
]
for column in NUMERIC_COLUMNS:
    market[column] = pd.to_numeric(market[column], errors='coerce')

valid_price_volume = (
    market['code'].notna()
    & np.isfinite(market['close'])
    & (market['close'] > 0)
    & np.isfinite(market['volume'])
    & (market['volume'] > 0)
    & np.isfinite(market['amount_thousand_yuan'])
    & (market['amount_thousand_yuan'] > 0)
    & np.isfinite(market['adj_factor'])
    & (market['adj_factor'] > 0)
)
valid_market_cap = (
    np.isfinite(market['total_market_cap'])
    & (market['total_market_cap'] > 0)
)

market['raw_vwap'] = (
    market['amount_thousand_yuan'] * 10.0 / market['volume']
).where(valid_price_volume)
market['adjusted_vwap'] = (
    market['raw_vwap'] * market['adj_factor']
).where(valid_price_volume)
valid_adjusted_vwap = (
    np.isfinite(market['adjusted_vwap'])
    & (market['adjusted_vwap'] > 0)
)
valid_pair = valid_price_volume & valid_adjusted_vwap
market['log_adjusted_vwap'] = np.log(
    market['adjusted_vwap'].where(valid_pair)
)
market['log_volume'] = np.log(
    market['volume'].where(valid_pair)
)
market['log_market_cap'] = np.log(
    market['total_market_cap'].where(valid_market_cap)
)

if EXCLUDE_ST:
    is_st = market['name'].fillna('').str.upper().str.contains('ST', regex=False)
else:
    is_st = pd.Series(False, index=market.index)

vwap_close_ratio = (
    market['raw_vwap'] / market['close']
).where(valid_pair)
ratio_quantiles = vwap_close_ratio.quantile([0.01, 0.50, 0.99])
assert 0.50 < ratio_quantiles.loc[0.50] < 1.50, 'VWAP单位换算疑似错误'

quality_summary = pd.Series({
    '记录行数': len(market),
    '有效价格成交量记录': int(valid_pair.sum()),
    '价格成交量异常记录': int((~valid_pair).sum()),
    '有效总市值记录': int(valid_market_cap.sum()),
    'ST记录': int(is_st.sum()),
    '重复股票日期': int(market.duplicated(['date', 'code']).sum()),
    'VWAP/收盘价 1%分位': ratio_quantiles.loc[0.01],
    'VWAP/收盘价中位数': ratio_quantiles.loc[0.50],
    'VWAP/收盘价 99%分位': ratio_quantiles.loc[0.99],
})
display(quality_summary.to_frame('值'))


## 4. 计算20日成交量价背离度原值

对每只股票分别计算固定20个观测的Pearson相关系数。实现使用组内累计和，但所有读取批次已经合并，因此不会在Parquet批次边界重置历史。

只有同时满足以下条件才保留：

- 20日内价格和成交量全部有效；
- 首尾全市场交易日序号相差19，保证没有停牌或缺失交易日；
- 两个序列在窗口内均有非零方差；
- 信号日不是ST。


In [ ]:
def rolling_corr_complete(x, y, window, eps=1e-12):
    """固定窗口相关系数；窗口内任何一个无效值都会使结果失效。"""
    x = np.asarray(x, dtype='float64')
    y = np.asarray(y, dtype='float64')
    n = len(x)
    result = np.full(n, np.nan, dtype='float64')
    counts = np.zeros(n, dtype='int16')
    if n < window:
        return result, counts

    valid = np.isfinite(x) & np.isfinite(y)
    x0 = np.where(valid, x, 0.0)
    y0 = np.where(valid, y, 0.0)

    def window_sum(values):
        prefix = np.concatenate(([0.0], np.cumsum(values, dtype='float64')))
        return prefix[window:] - prefix[:-window]

    count_prefix = np.concatenate(([0], np.cumsum(valid, dtype='int64')))
    count_window = count_prefix[window:] - count_prefix[:-window]
    sx = window_sum(x0)
    sy = window_sum(y0)
    sxx = window_sum(x0 * x0)
    syy = window_sum(y0 * y0)
    sxy = window_sum(x0 * y0)

    covariance = sxy - sx * sy / window
    variance_x = np.maximum(sxx - sx * sx / window, 0.0)
    variance_y = np.maximum(syy - sy * sy / window, 0.0)
    denominator = np.sqrt(variance_x * variance_y)
    usable = (
        (count_window == window)
        & (variance_x > eps)
        & (variance_y > eps)
        & np.isfinite(denominator)
    )
    corr = np.full(len(count_window), np.nan, dtype='float64')
    corr[usable] = covariance[usable] / denominator[usable]
    corr[usable] = np.clip(corr[usable], -1.0, 1.0)

    result[window - 1:] = corr
    counts[window - 1:] = count_window.astype('int16')
    return result, counts


rolling_corr = np.full(len(market), np.nan, dtype='float64')
rolling_valid_obs = np.zeros(len(market), dtype='int16')
group_indices = market.groupby('code', sort=False).indices

for number, (_, positions) in enumerate(group_indices.items(), start=1):
    corr_values, count_values = rolling_corr_complete(
        market['log_adjusted_vwap'].to_numpy()[positions],
        market['log_volume'].to_numpy()[positions],
        ROLLING_WINDOW,
        CORRELATION_EPS,
    )
    rolling_corr[positions] = corr_values
    rolling_valid_obs[positions] = count_values
    if number % 1000 == 0:
        print(f'已完成 {number:,}/{len(group_indices):,} 只股票')

market['rolling_valid_obs'] = rolling_valid_obs
market['price_volume_corr_20'] = rolling_corr

grouped_by_code = market.groupby('code', sort=False)
window_start_index = grouped_by_code['date_index'].shift(ROLLING_WINDOW - 1)
is_consecutive_window = (
    market['date_index'] - window_start_index
).eq(ROLLING_WINDOW - 1)
market.loc[~is_consecutive_window, 'price_volume_corr_20'] = np.nan

# 负相关越强，背离度越大。
market['volume_price_divergence_raw'] = -market['price_volume_corr_20']
market.loc[is_st, 'volume_price_divergence_raw'] = np.nan

finite_corr = market['price_volume_corr_20'].dropna()
assert not finite_corr.empty, '没有计算出有效滚动相关系数'
assert finite_corr.between(-1.0000001, 1.0000001).all(), '相关系数超出[-1, 1]'

target_mask = market['date'].between(START_TS, END_TS)
assert target_mask.any(), '正式训练区间内没有行情数据'
first_training_date = market.loc[target_mask, 'date'].min()
first_day_mask = market['date'].eq(first_training_date)
first_day_valid = (
    first_day_mask
    & ~is_st
    & market['rolling_valid_obs'].eq(ROLLING_WINDOW)
    & is_consecutive_window
    & market['volume_price_divergence_raw'].notna()
)
assert first_day_valid.sum() >= MIN_NEUTRALIZE_STOCKS, '训练首日有效因子数量不足'

# 固定抽样复算，验证累计和实现与np.corrcoef完全一致。
audit_rows = market.loc[
    target_mask & market['price_volume_corr_20'].notna(),
    ['date', 'code', 'price_volume_corr_20'],
].sample(n=5, random_state=20260824)
audit_results = []
for row in audit_rows.itertuples(index=False):
    history = market.loc[
        market['code'].eq(row.code) & market['date'].le(row.date),
        ['log_adjusted_vwap', 'log_volume'],
    ].tail(ROLLING_WINDOW)
    manual_corr = np.corrcoef(
        history['log_adjusted_vwap'], history['log_volume']
    )[0, 1]
    audit_results.append({
        'date': row.date,
        'code': row.code,
        'rolling_corr': row.price_volume_corr_20,
        'manual_corr': manual_corr,
        'abs_difference': abs(row.price_volume_corr_20 - manual_corr),
    })
rolling_audit = pd.DataFrame(audit_results)
assert rolling_audit['abs_difference'].max() < 1e-9, '滚动相关系数抽样复算失败'

display(pd.Series({
    '正式训练首日': first_training_date,
    '首日行情记录': int(first_day_mask.sum()),
    '首日有效背离度': int(first_day_valid.sum()),
    '有效相关系数最小值': finite_corr.min(),
    '有效相关系数最大值': finite_corr.max(),
    '抽样复算最大误差': rolling_audit['abs_difference'].max(),
}).to_frame('值'))
display(rolling_audit)

# 从这里开始只保留正式训练期；预热记录不参与截面处理和最终输出。
market = market.loc[target_mask].copy().reset_index(drop=True)
is_st = is_st.loc[target_mask].reset_index(drop=True)
is_consecutive_window = is_consecutive_window.loc[target_mask].reset_index(drop=True)
valid_pair = valid_pair.loc[target_mask].reset_index(drop=True)
assert market['date'].between(START_TS, END_TS).all()


## 5. 每日MAD去极值、市值中性化与标准化

只对正式训练期逐日执行：

1. 中位数 ± `3 × MAD` 去极值；
2. 回归 `VPD_winsor = 截距 + beta × log(总市值) + 残差`；
3. 将残差做每日Z-score，得到最终 `volume_price_divergence_factor`。

行情暂不包含完整行业和Barra风格暴露，因此只做可核验的Size中性化，不声称已经完成行业及全风格中性化。


In [ ]:
RAW_FACTOR_COLUMN = 'volume_price_divergence_raw'
WINSOR_COLUMN = 'volume_price_divergence_winsor'
NEUTRAL_COLUMN = 'volume_price_divergence_neutral'

date_group = market.groupby('date', sort=False)[RAW_FACTOR_COLUMN]
daily_median = date_group.transform('median')
absolute_deviation = (market[RAW_FACTOR_COLUMN] - daily_median).abs()
daily_mad = absolute_deviation.groupby(
    market['date'], sort=False
).transform('median')

lower_limit = daily_median - MAD_SCALE * daily_mad
upper_limit = daily_median + MAD_SCALE * daily_mad
clipped = market[RAW_FACTOR_COLUMN].clip(
    lower=lower_limit, upper=upper_limit
)
market[WINSOR_COLUMN] = clipped.where(
    daily_mad > 0, market[RAW_FACTOR_COLUMN]
)

eligible = (
    market[WINSOR_COLUMN].notna()
    & market['log_market_cap'].notna()
)
x = market['log_market_cap'].where(eligible)
y = market[WINSOR_COLUMN].where(eligible)
daily_key = market['date']
daily_n = eligible.groupby(daily_key, sort=False).transform('sum')
x_mean = x.groupby(daily_key, sort=False).transform('mean')
y_mean = y.groupby(daily_key, sort=False).transform('mean')
dx = x - x_mean
dy = y - y_mean
covariance_numerator = (dx * dy).groupby(
    daily_key, sort=False
).transform('sum')
x_variance_numerator = dx.pow(2).groupby(
    daily_key, sort=False
).transform('sum')
valid_regression = (
    (daily_n >= MIN_NEUTRALIZE_STOCKS)
    & (x_variance_numerator > 0)
)
size_beta = (covariance_numerator / x_variance_numerator).where(
    valid_regression
)
market[NEUTRAL_COLUMN] = (
    dy - size_beta * dx
).where(valid_regression)

neutral_group = market.groupby('date', sort=False)[NEUTRAL_COLUMN]
neutral_mean = neutral_group.transform('mean')
neutral_std = neutral_group.transform('std')
market[FACTOR_COLUMN] = (
    (market[NEUTRAL_COLUMN] - neutral_mean)
    / neutral_std.where(neutral_std > 0)
)

# 中性化诊断：残差与log(总市值)相关系数必须接近0。
y_sum_of_squares = dy.pow(2).groupby(
    daily_key, sort=False
).transform('sum')
raw_size_correlation = covariance_numerator / np.sqrt(
    x_variance_numerator * y_sum_of_squares
)
residual = market[NEUTRAL_COLUMN]
residual_centered = residual - residual.groupby(
    daily_key, sort=False
).transform('mean')
residual_covariance = (dx * residual_centered).groupby(
    daily_key, sort=False
).transform('sum')
residual_sum_of_squares = residual_centered.pow(2).groupby(
    daily_key, sort=False
).transform('sum')
neutralized_size_correlation = residual_covariance / np.sqrt(
    x_variance_numerator * residual_sum_of_squares
)

market['_size_beta'] = size_beta
market['_raw_size_corr'] = raw_size_correlation
market['_neutralized_size_corr'] = neutralized_size_correlation
market['_valid_pair'] = valid_pair.astype('int8')
market['_full_window'] = (
    market['rolling_valid_obs'].eq(ROLLING_WINDOW)
    & is_consecutive_window
).astype('int8')
market['_st_record'] = is_st.astype('int8')

factor_coverage = market.groupby('date').agg(
    market_rows=('code', 'size'),
    valid_price_volume=('_valid_pair', 'sum'),
    full_rolling_window=('_full_window', 'sum'),
    rolling_correlation=('price_volume_corr_20', 'count'),
    raw_factor=(RAW_FACTOR_COLUMN, 'count'),
    neutralized_factor=(NEUTRAL_COLUMN, 'count'),
    valid_factor=(FACTOR_COLUMN, 'count'),
    st_records=('_st_record', 'sum'),
    size_beta=('_size_beta', 'first'),
    raw_size_correlation=('_raw_size_corr', 'first'),
    neutralized_size_correlation=('_neutralized_size_corr', 'first'),
)
factor_coverage['coverage'] = (
    factor_coverage['valid_factor'] / factor_coverage['market_rows']
)

valid_neutral_corr = factor_coverage['neutralized_size_correlation'].dropna()
assert not valid_neutral_corr.empty, '没有可用的市值中性化结果'
assert valid_neutral_corr.abs().max() < 1e-7, '市值中性化检查失败'
assert factor_coverage['valid_factor'].min() >= MIN_NEUTRALIZE_STOCKS, (
    '存在整日有效因子数量不足'
)
assert (factor_coverage['raw_factor'] > 0).all(), '存在整日滚动因子异常缺失'

daily_factor_stats = market.groupby('date')[FACTOR_COLUMN].agg(['count', 'mean', 'std'])
assert daily_factor_stats['mean'].abs().max() < 1e-8, '每日Z-score均值检查失败'
assert (daily_factor_stats['std'] - 1.0).abs().max() < 1e-8, '每日Z-score标准差检查失败'

display(factor_coverage.describe().T)
display(pd.Series({
    '最低日有效因子数': factor_coverage['valid_factor'].min(),
    '最低日覆盖率': factor_coverage['coverage'].min(),
    '平均日覆盖率': factor_coverage['coverage'].mean(),
    '中性化后最大绝对市值相关': valid_neutral_corr.abs().max(),
    'Z-score最大绝对日均值': daily_factor_stats['mean'].abs().max(),
    'Z-score标准差最大偏差': (daily_factor_stats['std'] - 1.0).abs().max(),
}).to_frame('值'))


## 6. 因子分布和覆盖率诊断

图表只描述因子本身，不读取未来收益。重点检查覆盖率断点、相关系数是否长期漂移、分布是否集中在边界，以及市值中性化是否稳定。


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

CJK_FONT_CANDIDATES = [
    'Hiragino Sans GB', 'Arial Unicode MS', 'Heiti SC',
    'Songti SC', 'PingFang SC', 'Noto Sans CJK SC', 'SimHei',
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
CJK_FONT = next(
    (name for name in CJK_FONT_CANDIDATES if name in available_font_names),
    None,
)
if CJK_FONT:
    plt.rcParams['font.sans-serif'] = [CJK_FONT, 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

daily_raw = market.groupby('date')[RAW_FACTOR_COLUMN].agg(['mean', 'std'])
histogram_values = market[RAW_FACTOR_COLUMN].dropna()
if len(histogram_values) > 300_000:
    histogram_values = histogram_values.sample(300_000, random_state=20260824)

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
factor_coverage['coverage'].plot(ax=axes[0, 0], linewidth=1, color='tab:blue')
axes[0, 0].set_title('每日有效因子覆盖率')
axes[0, 0].set_ylim(0, 1.02)

daily_raw['mean'].plot(ax=axes[0, 1], linewidth=0.8, alpha=0.6, label='daily mean')
daily_raw['mean'].rolling(60, min_periods=20).mean().plot(
    ax=axes[0, 1], linewidth=1.2, label='60-day mean'
)
axes[0, 1].axhline(0, color='black', linewidth=0.8)
axes[0, 1].set_title('背离度原值的每日均值')
axes[0, 1].legend()

histogram_values.hist(ax=axes[1, 0], bins=60, color='tab:orange')
axes[1, 0].set_title('背离度原值分布（最多抽样30万条）')

factor_coverage[['raw_size_correlation', 'neutralized_size_correlation']].plot(
    ax=axes[1, 1], linewidth=0.9
)
axes[1, 1].axhline(0, color='black', linewidth=0.8)
axes[1, 1].set_title('市值相关性：处理前与中性化后')

for ax in axes.flat:
    ax.grid(alpha=0.25)
    ax.set_xlabel('')
plt.tight_layout()
diagnostic_chart = OUTPUT_DIR / f'volume_price_divergence_diagnostics_{DATE_TAG}.png'
fig.savefig(diagnostic_chart, dpi=160, bbox_inches='tight')
plt.show()
print('诊断图：', diagnostic_chart)


## 7. 保存标准因子Parquet及协议

最终文件只包含正式训练期，并保留原始相关系数、背离度原值、去极值值、中性化残差和标准化因子，便于以后审计。采用临时文件写入后原子替换，避免中途失败留下不完整Parquet。


In [ ]:
FACTOR_COLUMNS = [
    'date', 'code', 'name',
    'volume', 'adjusted_vwap', 'price_volume_corr_20',
    RAW_FACTOR_COLUMN, WINSOR_COLUMN, NEUTRAL_COLUMN, FACTOR_COLUMN,
]
factor_panel = market.loc[
    market[FACTOR_COLUMN].notna(), FACTOR_COLUMNS
].copy()
factor_panel['asof_date'] = factor_panel['date']

assert factor_panel['date'].between(START_TS, END_TS).all()
assert factor_panel['date'].min() == market['date'].min()
assert factor_panel['date'].max() == market['date'].max()
assert not factor_panel.duplicated(['date', 'code']).any()
assert np.isfinite(factor_panel[FACTOR_COLUMN]).all()
assert (factor_panel['asof_date'] <= factor_panel['date']).all()

factor_path = OUTPUT_DIR / f'volume_price_divergence_factor_{DATE_TAG}.parquet'
temporary_factor_path = factor_path.with_suffix('.tmp.parquet')
factor_panel.to_parquet(temporary_factor_path, index=False, compression='zstd')
temporary_factor_path.replace(factor_path)

coverage_path = OUTPUT_DIR / 'factor_coverage.csv'
temporary_coverage_path = coverage_path.with_suffix('.tmp.csv')
factor_coverage.to_csv(temporary_coverage_path, index_label='date')
temporary_coverage_path.replace(coverage_path)

protocol = {
    'factor_name': FACTOR_NAME,
    'factor_column': FACTOR_COLUMN,
    'factor_direction': FACTOR_DIRECTION,
    'preprocessed': True,
    'warmup_start': WARMUP_START_DATE,
    'sample_start': START_DATE,
    'sample_end': END_DATE,
    'market_exchanges': list(MARKET_EXCHANGES),
    'rolling_window': ROLLING_WINDOW,
    'min_periods': MIN_PERIODS,
    'requires_consecutive_market_days': True,
    'adjusted_vwap_formula': 'amount_thousand_yuan * 10 / volume_hands * adj_factor',
    'raw_definition': '-corr(log(adjusted_vwap), log(volume), 20 complete days)',
    'winsorization': f'daily median +/- {MAD_SCALE} * MAD',
    'neutralization': 'daily OLS residual on log(total_market_cap)',
    'standardization': 'daily z-score of neutralized residual',
    'exclude_st_on_signal_date': EXCLUDE_ST,
    'factor_available_at': 'signal_date close',
    'earliest_execution': 'signal_date + 1 trading day',
    'output_path': str(factor_path),
}
protocol_path = OUTPUT_DIR / 'factor_protocol.json'
temporary_protocol_path = protocol_path.with_suffix('.tmp.json')
temporary_protocol_path.write_text(
    json.dumps(protocol, ensure_ascii=False, indent=2), encoding='utf-8'
)
temporary_protocol_path.replace(protocol_path)

saved_schema = pq.read_schema(factor_path)
saved_rows = pq.ParquetFile(factor_path).metadata.num_rows
assert saved_rows == len(factor_panel)
assert {'date', 'code', FACTOR_COLUMN, 'asof_date'}.issubset(saved_schema.names)

factor_summary = pd.Series({
    '因子记录数': len(factor_panel),
    '股票数量': factor_panel['code'].nunique(),
    '有效因子首日': factor_panel['date'].min(),
    '有效因子末日': factor_panel['date'].max(),
    '最低日覆盖率': factor_coverage['coverage'].min(),
    '中性化后最大绝对市值相关': valid_neutral_corr.abs().max(),
    '输出大小MB': factor_path.stat().st_size / 1024 / 1024,
    '冻结回测方向': FACTOR_DIRECTION,
})
display(factor_summary.to_frame('值'))
print('因子文件：', factor_path)
print('覆盖率：', coverage_path)
print('因子协议：', protocol_path)


## 8. 接入通用回测

在 `Back_test/通用因子回测模板.ipynb` 中冻结为：

```python
FACTOR_NAME = 'volume_price_divergence'
FACTOR_COLUMN = 'volume_price_divergence_factor'
FACTOR_DIRECTION = 'negative'
FACTOR_PREPROCESSED = True

FACTOR_SOURCE = (
    LEARNING_DIR / 'Factor'
    / 'volume_price_divergence_output_20060104_20171229'
    / 'volume_price_divergence_factor_20060104_20171229.parquet'
)

HORIZON = 5
EXECUTION_LAG = 1
REBALANCE_STEP = 5
```

最终因子已经完成MAD、Size中性化和Z-score，因此通用回测中保持 `NEUTRALIZE=False`，避免重复处理。

训练回测前已经冻结“背离越强越偏空”的方向。若训练结果不支持该方向，应记录为因子失败或另立新假设，不能仅为了让结果变好而无记录地翻转方向。验证阶段只修改因子生产日期与输出路径，公式和全部参数保持一致。
